# Color palette extraction

In this script,
we compute a subsampled color palette time series for an input video file.
In non-technical terms,
every few frames,
we compute the current dominant colors and store them sequentially throughout the entire video.

Specifically,
the core palette extraction step is done using $k$-means clustering:
for a given subsampled frame,
we represent each pixel in BGR space (by its blue, green, and red content);
then all the pixels form a 3-D point cloud.
The $k$-means clustering algorithm then tries to label each clusters of points and returns the cluster centers—the colors that are representative of their peers in the same cluster, and are somewhat distinct with each other across clusters.

In [ ]:
# import libraries
import os
import pathlib

import numpy as np
import sklearn
import tqdm

from video import Video

In [ ]:
# define parameters
# We sample every STRIDE=6 frames, and for each subsampled frame we identify
# NUM_CLUSTERS=6 palette colors.

STRIDE = 6
NUM_CLUSTERS = 6
RANDOM_STATE = 0
NAME = "palettes"

In [ ]:
# specify file structure
base_dir = pathlib.Path(os.getcwd())
input_dir = base_dir / "input"
output_dir = base_dir / "output"

In [ ]:
# read video
video = Video(input_dir / "perfect_blue.mp4", verbose=True)

In [ ]:
# perform k-means clustering for each frame
# This takes around three hours.

out = np.empty((video.num_frames // STRIDE, NUM_CLUSTERS, 3), dtype=np.uint8)

for i in tqdm.trange(video.num_frames // STRIDE):
    frame = video[STRIDE * i]
    
    X = frame.reshape(-1, 3)
    kmeans = sklearn.cluster.KMeans(
        n_clusters=NUM_CLUSTERS,
        random_state=RANDOM_STATE,
    ).fit(X)
    
    centers = np.clip(kmeans.cluster_centers_.astype(np.uint8), 0, 255)
    out[i] = centers

In [ ]:
# save palettes
np.save(output_dir / NAME, out)